# Day 1: Basic Python API with FastAPI

## Goal

Build a small API that exposes one endpoint, `/query`, and supports simple CRUD operations.

By the end of this notebook you will have:

- a working FastAPI app
- one `/query` endpoint
- simple in-memory CRUD logic
- example requests that prove the API works

This notebook uses **FastAPI** because it is concise, strongly typed, and a good fit for AI-facing backend services.

## What We Are Building

We will store simple `query` records in memory using a Python dictionary.

Each record will have:

- `id`
- `question`
- `answer`
- `status`

The single `/query` endpoint will handle these actions:

- `create`
- `read`
- `update`
- `delete`
- `list`

Using one endpoint is not ideal for production REST design, but it keeps Day 1 focused and maps directly to the training goal.

## Step 0: Sync the Project Once

The Day 1 dependencies are already included in `pyproject.toml`, so learners only need to run:

```bash
uv sync
```

Then make sure the notebook is using the project `.venv` kernel.

For course maintainers, the dependencies were added with:

```bash
uv add fastapi "uvicorn[standard]" httpx
```

In [ ]:
from typing import Literal

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

print("Imports loaded successfully.")

## Step 1: Define the Request Model

We use Pydantic models to validate incoming JSON.

The client will send an `action` plus the fields needed for that action.

In [ ]:
class QueryRequest(BaseModel):
    action: Literal["create", "read", "update", "delete", "list"]
    id: str | None = Field(default=None, description="Unique id for a stored query")
    question: str | None = None
    answer: str | None = None
    status: str | None = None


QueryRequest.model_json_schema()

## Step 2: Build the API

We will keep the data in a plain dictionary for now. That makes the CRUD logic easy to understand before introducing databases later in the course.

In [ ]:
app = FastAPI(title="Day 1 Query API")

# In-memory storage for Day 1.
query_store: dict[str, dict] = {}


@app.post("/query")
def handle_query(request: QueryRequest):
    if request.action == "create":
        if not request.id:
            raise HTTPException(status_code=400, detail="'id' is required for create")
        if request.id in query_store:
            raise HTTPException(status_code=409, detail="Query id already exists")

        query_store[request.id] = {
            "id": request.id,
            "question": request.question or "",
            "answer": request.answer or "",
            "status": request.status or "new",
        }
        return {"message": "Query created", "data": query_store[request.id]}

    if request.action == "read":
        if not request.id:
            raise HTTPException(status_code=400, detail="'id' is required for read")
        if request.id not in query_store:
            raise HTTPException(status_code=404, detail="Query not found")
        return {"data": query_store[request.id]}

    if request.action == "update":
        if not request.id:
            raise HTTPException(status_code=400, detail="'id' is required for update")
        if request.id not in query_store:
            raise HTTPException(status_code=404, detail="Query not found")

        current = query_store[request.id]
        if request.question is not None:
            current["question"] = request.question
        if request.answer is not None:
            current["answer"] = request.answer
        if request.status is not None:
            current["status"] = request.status

        return {"message": "Query updated", "data": current}

    if request.action == "delete":
        if not request.id:
            raise HTTPException(status_code=400, detail="'id' is required for delete")
        if request.id not in query_store:
            raise HTTPException(status_code=404, detail="Query not found")

        deleted = query_store.pop(request.id)
        return {"message": "Query deleted", "data": deleted}

    return {"count": len(query_store), "data": list(query_store.values())}


print("FastAPI app created.")

## Step 3: Test the Endpoint Inside the Notebook

FastAPI ships with a test client. This lets us exercise the API immediately without leaving Jupyter.

In [ ]:
client = TestClient(app)


def call_api(payload: dict):
    response = client.post("/query", json=payload)
    print(f"Status: {response.status_code}")
    print(response.json())
    print("-" * 80)


print("Test client ready.")

### Create

In [ ]:
call_api(
    {
        "action": "create",
        "id": "query-001",
        "question": "What is FastAPI?",
        "answer": "A modern Python framework for building APIs.",
        "status": "open",
    }
)

### Read

In [ ]:
call_api({"action": "read", "id": "query-001"})

### Update

In [ ]:
call_api(
    {
        "action": "update",
        "id": "query-001",
        "answer": "A modern Python framework for APIs with validation and docs.",
        "status": "resolved",
    }
)

### List

In [ ]:
call_api({"action": "list"})

### Delete

In [ ]:
call_api({"action": "delete", "id": "query-001"})

## Optional: Run It as a Local API Server

If you want to expose this app outside the notebook, move the API code into a Python file such as `api_app.py` and run:

```bash
uvicorn api_app:app --reload
```

Then send a POST request to `http://127.0.0.1:8000/query`.

Example body:

```json
{
  "action": "create",
  "id": "query-002",
  "question": "Explain CRUD",
  "answer": "Create, Read, Update, Delete",
  "status": "new"
}
```

## Day 1 Recap

You now have a basic Python API with:

- FastAPI setup
- request validation
- one `/query` endpoint
- simple CRUD behavior
- notebook-based testing

Day 2 can build directly on this by replacing the static `answer` field with a real LLM call.